In [ ]:
from datasets import load_dataset

ds = load_dataset("Genius-Society/Pima")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/614 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/77 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/77 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
# Extraire les features dans un tableau numpy
X = np.array([[s['Pregnancies'], s['Glucose'], s['BloodPressure'], s['SkinThickness'],
               s['Insulin'], s['BMI'], s['DiabetesPedigreeFunction'], s['Age']]
              for s in ds['train']])

def density_encoding_real(x):
    # Normaliser le vecteur
    x_norm = x / np.linalg.norm(x)
    # Créer l'état pur
    psi = x_norm.reshape(-1, 1)  # colonne
    # Matrice de densité : rho = |psi><psi|
    rho = np.dot(psi, psi.conj().T)
    return rho

# Exemple sur la première ligne du dataset
rho_example = density_encoding_real(X[0])
print("Density Encoding (real, matrix):")
print(rho_example)


Density Encoding (real, matrix):
[[2.58547460e-04 1.10529039e-02 4.65385428e-03 1.87446909e-03
  1.00187141e-02 2.81816732e-03 3.09610583e-05 1.68055849e-03]
 [1.10529039e-02 4.72511643e-01 1.98952271e-01 8.01335534e-02
  4.28300027e-01 1.20476653e-01 1.32358524e-03 7.18438755e-02]
 [4.65385428e-03 1.98952271e-01 8.37693771e-02 3.37404435e-02
  1.80336853e-01 5.07270117e-02 5.57299050e-04 3.02500528e-02]
 [1.87446909e-03 8.01335534e-02 3.37404435e-02 1.35899009e-02
  7.26356771e-02 2.04317130e-02 2.24467673e-04 1.21840491e-02]
 [1.00187141e-02 4.28300027e-01 1.80336853e-01 7.26356771e-02
  3.88225171e-01 1.09203983e-01 1.19974101e-03 6.51216415e-02]
 [2.81816732e-03 1.20476653e-01 5.07270117e-02 2.04317130e-02
  1.09203983e-01 3.07180237e-02 3.37475536e-04 1.83180875e-02]
 [3.09610583e-05 1.32358524e-03 5.57299050e-04 2.24467673e-04
  1.19974101e-03 3.37475536e-04 3.70758674e-06 2.01246879e-04]
 [1.68055849e-03 7.18438755e-02 3.02500528e-02 1.21840491e-02
  6.51216415e-02 1.83180875e-0

In [3]:
# Appliquer l'encodage à tout le dataset
# X_rho sera de forme (n_samples, n_features, n_features)
X_rho = np.array([density_encoding_real(row) for row in X])

print(f"Forme du dataset original : {X.shape}")
print(f"Forme du dataset encodé (Density) : {X_rho.shape}")
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

# 1. On aplatit les matrices (768, 8, 8) -> (768, 64)
X_flat = X_rho.reshape(len(X_rho), -1)

# 2. Récupération des étiquettes (labels)
y = np.array([s['Outcome'] for s in ds['train']])

# 3. Séparation Entraînement / Test
X_train, X_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=42)
# Initialisation et entraînement
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

# Prédiction et score
y_pred = clf.predict(X_test)
print(f"Précision de l'arbre avec Density Encoding : {accuracy_score(y_test, y_pred):.2%}")

Forme du dataset original : (614, 8)
Forme du dataset encodé (Density) : (614, 8, 8)
Précision de l'arbre avec Density Encoding : 67.48%
